In [13]:
import pandas as pd

df = pd.read_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\6번.데이터채우기후\M19_도매_소매업.parquet')

## 1. 설립일 (날짜형 데이터)

In [14]:
BASE_YEAR = 2024

df['설립일'] = pd.to_datetime(df['설립일'], errors='coerce')
df['업력']   = BASE_YEAR - df['설립일'].dt.year

nan_before = df['업력'].isna().sum()
print(f'\n업력 생성 후 NaN: {nan_before:,}개')

# 음수 → NaN (설립일이 2024년 이후거나 오류인 기업)
neg_mask = df['업력'] < 0
print(f'음수 업력 → NaN 처리: {neg_mask.sum():,}건')
df.loc[neg_mask, '업력'] = float('nan')

print(f'업력 NaN (최종): {df["업력"].isna().sum():,}개')
print(f'업력 범위: {df["업력"].min():.0f}년 ~ {df["업력"].max():.0f}년')

# 설립일 컬럼 제거 (업력으로 변환 완료)
df.drop(columns=['설립일'], inplace=True)
print('설립일 → 업력 변환 완료')



업력 생성 후 NaN: 27개
음수 업력 → NaN 처리: 0건
업력 NaN (최종): 27개
업력 범위: 2년 ~ 105년
설립일 → 업력 변환 완료


In [15]:
df['업력'].describe()

count    39881.000000
mean        22.160879
std         11.482327
min          2.000000
25%         14.000000
50%         21.000000
75%         28.000000
max        105.000000
Name: 업력, dtype: float64

In [16]:
neg = df[df["업력"] < 0]

print("음수 업력 건수:", len(neg))

print(
    neg[
        ["회사명", "회계년도", "업력"]
    ].head(30)
)

음수 업력 건수: 0
Empty DataFrame
Columns: [회사명, 회계년도, 업력]
Index: []


설립일 컬럼 검증 결과, 회계연도보다 미래 시점의 설립일이 다수 존재하여 음수 업력이 발생하였다. 또한 DART 정보와의 불일치 및 결측치가 확인되어 데이터 신뢰성이 낮다고 판단하였으며, 최종 분석에서는 해당 컬럼(업력-설립일)을 제외하였다.

## 2. 외부감사기관 (범주형 데이터)

In [17]:
# ── 2. 외부감사기관 → 빅4 더미변수 ─────────────────────────

BIG4 = ["삼일", "삼정", "한영", "안진"]

df["빅4감사"] = df["외부감사기관"].apply(
    lambda x: (
        1
        if pd.notna(x)
        and any(b in str(x) for b in BIG4)
        else 0
    )
).astype(int)



print("\n✅ 외부감사기관 → 빅4감사 더미변수 변환 완료")
print(
    df["빅4감사"]
      .value_counts()
      .rename({1: "빅4(1)", 0: "비빅4(0)"})
      .to_string()
)

df.drop(columns=["외부감사기관"], inplace=True)


✅ 외부감사기관 → 빅4감사 더미변수 변환 완료
빅4감사
비빅4(0)    34515
빅4(1)      5393


## 3. 기존 비율에 x 100 하는 단위변환 처리 (산업별 평균 데이터와의 단위 비슷하게 하기 위한 목적)

In [18]:
df.loc[:, '부채비율':].describe()

,부채비율,총부채비율,장기부채비율,장기부채의존도,차입금의존도,순차입금비율,금융부채비율,자기자본비율,유보율,자본잠식률,...,부채증가율,영업현금흐름증가율,FCF증가율,ROA변화,영업이익률변화,부채비율변화,유동비율변화,부실라벨_ICR3년,업력,빅4감사
count,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,...,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,39881.000000,39908.000000
mean,11.741374,0.623768,2.786849,0.132449,0.327025,6.781667,7.349521,0.376232,65.798346,-67.575046,...,0.384676,1.012892,0.212065,-0.007500,-0.196263,0.185726,0.247957,0.037812,22.160879,0.135136
std,192.907456,2.375604,65.997795,0.324781,1.841435,138.978538,141.672283,2.375606,304.838142,307.003536,...,39.822839,35.192767,53.920119,0.117252,38.534786,239.349907,69.074760,0.190744,11.482327,0.341873
min,-0.062804,-0.067013,-0.225532,-0.015969,0.000000,-128.736667,0.000000,-382.411765,-10267.000000,-23293.000000,...,-1.000000,-1192.000000,-5202.166667,-3.193748,-6849.000000,-30503.998088,-7446.958112,0.000000,2.000000,0.000000
25%,0.622814,0.383786,0.024274,0.009919,0.040526,-0.054344,0.082315,0.222491,3.080000,-51.581350,...,-0.086221,-0.088424,0.000000,-0.022165,-0.016200,-0.251142,-0.146785,0.000000,14.000000,0.000000
50%,1.581808,0.612667,0.157997,0.056397,0.270283,0.527895,0.699422,0.387322,14.152667,-15.244358,...,0.000000,0.000000,0.000000,0.000000,0.000000,-0.003197,0.000000,0.000000,21.000000,0.000000
75%,3.494523,0.777478,0.653737,0.174310,0.501618,1.754478,1.941124,0.616213,49.961250,-3.566895,...,0.162783,0.140841,0.043020,0.009959,0.009572,0.084056,0.164115,0.000000,28.000000,0.000000
max,30532.000000,383.411765,11828.000000,30.751149,300.764706,24648.000000,24677.000000,1.067013,24919.000000,10659.400000,...,7926.000000,3175.666667,7037.500000,6.408443,2778.000000,30459.597619,7443.399505,1.000000,105.000000,1.000000


In [19]:
"""
Cell output 14 [DW] 단위 변환 코드
=====================================
- 비율/이익률/증가율 계열: 소수(비율) → % 단위로 × 100
- 회전율 계열: 이미 "회" 단위로 동일 → 변환 없음
- 산업평균과 매핑 불가 컬럼(라벨, 메타): 변환 없음
"""

# ─────────────────────────────────────────────
# 2. × 100 변환이 필요한 컬럼 정의
#    (소수 형태 → % 형태로 통일)
# ─────────────────────────────────────────────

# [자산/자본 비율 계열]
asset_ratio_cols = [
    "부채비율",           # 소수 → %   예) 1.693 → 169.3%
    "총부채비율",         # 소수 → %
    "장기부채비율",       # 소수 → %
    "장기부채의존도",     # 소수 → %
    "차입금의존도",       # 소수 → %   예) 0.352 → 35.2%
    "순차입금비율",       # 소수 → %
    "금융부채비율",       # 소수 → %
    "자기자본비율",       # 소수 → %   예) 0.371 → 37.1%
    "유보율",             # 소수 → %
    "자본잠식률",         # 소수 → %
    "유동비율",           # 소수 → %   예) 1.248 → 124.8%
    "당좌비율_추정",      # 소수 → %   산업평균 '당좌비율'에 대응
    "현금비율",           # 소수 → %   예) 0.132 → 13.2%
    "순운전자본비율",     # 소수 → %
    "비유동비율",         # 소수 → %   예) 1.241 → 124.1%
    "비유동장기적합률",   # 소수 → %   예) 0.852 → 85.2%
    "유형자산비율",       # 소수 → %
    "유형자산부채비율",   # 소수 → %
]

# [수익성 지표 계열]
profitability_cols = [
    "ROA",                # 소수 → %   예) 0.017 → 1.7%
    "ROE",                # 소수 → %   예) 0.104 → 10.4%
    "ROIC",               # 소수 → %
    "총자본영업이익률",   # 소수 → %
    "매출총이익률",       # 소수 → %
    "영업이익률",         # 소수 → %   산업평균 '매출액영업이익률'에 대응
    "순이익률",           # 소수 → %   산업평균 '매출액순이익률'에 대응
    "EBITDA마진",         # 소수 → %   산업평균 'EBITDA대매출액'에 대응
    "영업현금흐름비율",   # 소수 → %
    "현금ROA",            # 소수 → %
    "현금ROE",            # 소수 → %
    "매출원가율",         # 소수 → %   산업평균 '매출원가대매출액'에 대응
    "판관비율",           # 소수 → %
    "감가상각비율",       # 소수 → %   산업평균 '감가상각률'에 대응
    "금융비용부담률",     # 소수 → %   산업평균 '금융비용대매출액'에 대응
    "영업CF_유동부채",    # 소수 → %
    "영업CF_총부채",      # 소수 → %
    "FCF_총자산",         # 소수 → %
]

# [성장성 지표 계열]
growth_cols = [
    "매출액증가율",       # 소수 → %   예) 0.05 → 5%
    "영업이익증가율",     # 소수 → %
    "순이익증가율",       # 소수 → %
    "EBITDA증가율",       # 소수 → %
    "총자산증가율",       # 소수 → %   예) 0.017 → 1.7%
    "유형자산증가율",     # 소수 → %
    "자기자본증가율",     # 소수 → %
    "부채증가율",         # 소수 → %
    "영업현금흐름증가율", # 소수 → %
    "FCF증가율",          # 소수 → %
]

# [변화량 지표 계열]
change_cols = [
    "ROA변화",            # 소수 → %p
    "영업이익률변화",     # 소수 → %p
    "부채비율변화",       # 소수 → %p
    "유동비율변화",       # 소수 → %p
]

# 전체 × 100 대상 컬럼 통합
cols_to_multiply = (
    asset_ratio_cols
    + profitability_cols
    + growth_cols
    + change_cols
)

# ─────────────────────────────────────────────
# 3. 변환 적용 (× 100)
# ─────────────────────────────────────────────
df_converted = df.copy()

for col in cols_to_multiply:
    if col in df_converted.columns:
        df_converted[col] = df_converted[col] * 100
    else:
        print(f"[WARNING] '{col}' 컬럼이 데이터에 없습니다. 건너뜁니다.")

# ─────────────────────────────────────────────
# 4. 변환 안 하는 컬럼 목록 (참고용)
# ─────────────────────────────────────────────
#  회전율 계열: 이미 "회(×)" 단위로 산업평균과 동일
#    - 총자산회전율, 유동자산회전율, 비유동자산회전율
#    - 유형자산회전율, 자기자본회전율, 투하자본회전율
#    - 매출채권회전율, 재고자산회전율, 매입채무회전율
#    - 순운전자본회전율
#  기간(일) 계열: 별도 단위(일)
#    - 매출채권회수기간, 재고자산보유기간, 매입채무지급기간, 현금전환주기_CCC
#  절대금액 계열
#    - FCF
#  이자보상배율: 배수(×) 단위
#  라벨/메타: 부실라벨_ICR3년, 빅4감사

# ─────────────────────────────────────────────
# 5. 변환 결과 검증 (median 기준)
# ─────────────────────────────────────────────
print("=" * 65)
print("변환 결과 검증 (median 행 기준, 산업평균 범위와 비교)")
print("=" * 65)

# CSV는 describe() 결과: row0=count, row1=mean, row2=std, row3=min, row4=25%, row5=median
median_before = df.iloc[5]
median_after  = df_converted.iloc[5]

validation = {
    "부채비율":         ("99~116%",    "자산자본"),
    "자기자본비율":     ("46~52%",     "자산자본"),
    "유동비율":         ("105~162%",   "자산자본"),
    "당좌비율_추정":    ("74~109%",    "자산자본 / '당좌비율'"),
    "현금비율":         ("12~22%",     "자산자본"),
    "비유동비율":       ("92~131%",    "자산자본"),
    "비유동장기적합률": ("70~97%",     "자산자본"),
    "차입금의존도":     ("27~30%",     "자산자본"),
    "ROA":              ("2~8%",       "수익성"),
    "ROE":              ("5~15%",      "수익성"),
    "영업이익률":       ("6~8%",       "손익 / '매출액영업이익률'"),
    "순이익률":         ("3~7%",       "손익 / '매출액순이익률'"),
    "EBITDA마진":       ("10~12%",     "손익 / 'EBITDA대매출액'"),
    "매출원가율":       ("67~68%",     "손익 / '매출원가대매출액'"),
    "감가상각비율":     ("8~10%",      "손익 / '감가상각률'"),
    "금융비용부담률":   ("0.98~1.52%", "손익 / '금융비용대매출액'"),
    "매출액증가율":     ("2~4%",       "성장성"),
    "총자산증가율":     ("2~8%",       "성장성"),
    "유형자산증가율":   ("1~8%",       "성장성"),
    "자기자본증가율":   ("3~9%",       "성장성"),
}

print(f"{'컬럼':<20} {'변환전':>10} {'변환후(×100)':>14}  {'산업평균 범위':<15} {'대응 카테고리'}")
print("-" * 90)
for col, (ref, category) in validation.items():
    before = median_before[col]
    after  = median_after[col]
    print(f"{col:<20} {before:>10.4f} {after:>14.4f}  {ref:<15} {category}")


print(f"   전체 행 수: {len(df_converted)}, 컬럼 수: {len(df_converted.columns)}")
print(f"   × 100 적용 컬럼 수: {len([c for c in cols_to_multiply if c in df_converted.columns])}개")

변환 결과 검증 (median 행 기준, 산업평균 범위와 비교)
컬럼                          변환전      변환후(×100)  산업평균 범위         대응 카테고리
------------------------------------------------------------------------------------------
부채비율                     0.5856        58.5603  99~116%         자산자본
자기자본비율                   0.6307        63.0675  46~52%          자산자본
유동비율                     1.9115       191.1505  105~162%        자산자본
당좌비율_추정                  1.0018       100.1798  74~109%         자산자본 / '당좌비율'
현금비율                     0.1342        13.4236  12~22%          자산자본
비유동비율                    0.7408        74.0804  92~131%         자산자본
비유동장기적합률                 0.6478        64.7754  70~97%          자산자본
차입금의존도                   0.2728        27.2767  27~30%          자산자본
ROA                      0.0206         2.0566  2~8%            수익성
ROE                      0.0325         3.2499  5~15%           수익성
영업이익률                    0.1127        11.2699  6~8%            손익 / '매출액영업이익률'
순이익률                    

In [20]:
df_converted.loc[:, '부채비율':].describe()

,부채비율,총부채비율,장기부채비율,장기부채의존도,차입금의존도,순차입금비율,금융부채비율,자기자본비율,유보율,자본잠식률,...,부채증가율,영업현금흐름증가율,FCF증가율,ROA변화,영업이익률변화,부채비율변화,유동비율변화,부실라벨_ICR3년,업력,빅4감사
count,3.990800e+04,39908.000000,3.990800e+04,39908.000000,39908.000000,3.990800e+04,3.990800e+04,39908.000000,3.990800e+04,3.990800e+04,...,39908.000000,39908.000000,39908.000000,39908.000000,39908.000000,3.990800e+04,39908.000000,39908.000000,39881.000000,39908.000000
mean,1.174137e+03,62.376811,2.786849e+02,13.244863,32.702496,6.781667e+02,7.349521e+02,37.623201,6.579835e+03,-6.757505e+03,...,38.467561,101.289231,21.206502,-0.750017,-19.626291,1.857259e+01,24.795717,0.037812,22.160879,0.135136
std,1.929075e+04,237.560441,6.599780e+03,32.478090,184.143470,1.389785e+04,1.416723e+04,237.560550,3.048381e+04,3.070035e+04,...,3982.283870,3519.276748,5392.011869,11.725158,3853.478613,2.393499e+04,6907.476024,0.190744,11.482327,0.341873
min,-6.280448e+00,-6.701321,-2.255319e+01,-1.596867,0.000000,-1.287367e+04,0.000000e+00,-38241.176471,-1.026700e+06,-2.329300e+06,...,-100.000000,-119200.000000,-520216.666667,-319.374821,-684900.000000,-3.050400e+06,-744695.811241,0.000000,2.000000,0.000000
25%,6.228141e+01,38.378649,2.427442e+00,0.991941,4.052650,-5.434393e+00,8.231469e+00,22.249083,3.080000e+02,-5.158135e+03,...,-8.622129,-8.842359,0.000000,-2.216509,-1.620038,-2.511418e+01,-14.678522,0.000000,14.000000,0.000000
50%,1.581808e+02,61.266746,1.579966e+01,5.639712,27.028268,5.278953e+01,6.994222e+01,38.732185,1.415267e+03,-1.524436e+03,...,0.000000,0.000000,0.000000,0.000000,0.000000,-3.196580e-01,0.000000,0.000000,21.000000,0.000000
75%,3.494523e+02,77.747792,6.537372e+01,17.430963,50.161803,1.754478e+02,1.941124e+02,61.621342,4.996125e+03,-3.566895e+02,...,16.278268,14.084062,4.301987,0.995895,0.957213,8.405605e+00,16.411502,0.000000,28.000000,0.000000
max,3.053200e+06,38341.176471,1.182800e+06,3075.114855,30076.470588,2.464800e+06,2.467700e+06,106.701321,2.491900e+06,1.065940e+06,...,792600.000000,317566.666667,703750.000000,640.844273,277800.000000,3.045960e+06,744339.950526,1.000000,105.000000,1.000000


In [21]:
df = df_converted

# 원본 재무지표 제거

In [22]:
cols_to_drop = [

    # 원본 재무지표
    #'자산총계(요약)(백만원)',
    '유동자산(요약)(백만원)',
    '현금 및 현금성자산(요약)(백만원)',
    '매출채권(요약)(백만원)',
    '재고자산(요약)(백만원)',
    '비유동자산(요약)(백만원)',
    '유형자산(요약)(백만원)',
    '부채총계(요약)(백만원)',
    '유동부채(요약)(백만원)',
    '매입채무(요약)(백만원)',
    '단기차입금(요약)(백만원)',
    '유동성장기부채(요약)(백만원)',
    '사채(요약)(백만원)',
    '장기차입금(요약)(백만원)',
    '비유동부채(요약)(백만원)',
    '자본총계(요약)(백만원)',
    '자본금(요약)(백만원)',
    '자본잉여금(요약)(백만원)',
    '이익잉여금(요약)(백만원)',
    '매출액(요약)(백만원)',
    '매출원가(요약)(백만원)',
    '매출총이익(요약)(백만원)',
    '판매비와 관리비(요약)(백만원)',
    '영업이익(요약)(백만원)',
    '이자비용(요약)(백만원)',
    '당기순이익(요약)(백만원)',
    '법인세비용차감전(계속사업)손익(요약)(백만원)',
    '(계속사업손익)법인세비용(요약)(백만원)',
    '영업활동으로 인한 현금흐름(요약)(백만원)',
    '*감가상각비',

    # 쓸모 없는 데이터 제거 (타겟 인코딩 했기에 제거함)
    '이자보상배율',

    # 사업자등록번호와 유사해서 데이터 제거
    '금감원등록번호'
]

df = df.drop(columns=cols_to_drop)

## 저장

In [23]:
df.to_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\7번\M19_도매_소매업.parquet')

In [24]:
df.shape

(39908, 79)